# EBAC - Regressão II - regressão múltipla

## Tarefa I

#### Previsão de renda

Vamos trabalhar com a base 'previsao_de_renda.csv', que é a base do seu próximo projeto. Vamos usar os recursos que vimos até aqui nesta base.

|variavel|descrição|
|-|-|
|data_ref                | Data de referência de coleta das variáveis |
|index                   | Código de identificação do cliente|
|sexo                    | Sexo do cliente|
|posse_de_veiculo        | Indica se o cliente possui veículo|
|posse_de_imovel         | Indica se o cliente possui imóvel|
|qtd_filhos              | Quantidade de filhos do cliente|
|tipo_renda              | Tipo de renda do cliente|
|educacao                | Grau de instrução do cliente|
|estado_civil            | Estado civil do cliente|
|tipo_residencia         | Tipo de residência do cliente (própria, alugada etc)|
|idade                   | Idade do cliente|
|tempo_emprego           | Tempo no emprego atual|
|qt_pessoas_residencia   | Quantidade de pessoas que moram na residência|
|renda                   | Renda em reais|

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('previsao_de_renda.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             15000 non-null  int64  
 1   data_ref               15000 non-null  object 
 2   id_cliente             15000 non-null  int64  
 3   sexo                   15000 non-null  object 
 4   posse_de_veiculo       15000 non-null  bool   
 5   posse_de_imovel        15000 non-null  bool   
 6   qtd_filhos             15000 non-null  int64  
 7   tipo_renda             15000 non-null  object 
 8   educacao               15000 non-null  object 
 9   estado_civil           15000 non-null  object 
 10  tipo_residencia        15000 non-null  object 
 11  idade                  15000 non-null  int64  
 12  tempo_emprego          12427 non-null  float64
 13  qt_pessoas_residencia  15000 non-null  float64
 14  renda                  15000 non-null  float64
dtypes:

1. Ajuste um modelo para prever log(renda) considerando todas as covariáveis disponíveis.
    - Utilizando os recursos do Patsy, coloque as variáveis qualitativas como *dummies*.
    - Mantenha sempre a categoria mais frequente como casela de referência
    - Avalie os parâmetros e veja se parecem fazer sentido prático.  


2. Remova a variável menos significante e analise:
    - Observe os indicadores que vimos, e avalie se o modelo melhorou ou piorou na sua opinião.
    - Observe os parâmetros e veja se algum se alterou muito.  


3. Siga removendo as variáveis menos significantes, sempre que o *p-value* for menor que 5%. Compare o modelo final com o inicial. Observe os indicadores e conclua se o modelo parece melhor. 
    

In [7]:
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrices
import matplotlib.pyplot as plt
import warnings

# Ignorar warnings para melhor visualização (cuidado ao usar em produção)
warnings.filterwarnings('ignore')

print("--- Exercício: Previsão de Renda - Regressão Múltipla ---")

# Carregar a base de dados
try:
    df = pd.read_csv('previsao_de_renda.csv')
    print("Base de dados 'previsao_de_renda.csv' carregada com sucesso!")
except FileNotFoundError:
    print("Erro: 'previsao_de_renda.csv' não encontrado. Por favor, verifique o caminho do arquivo.")
    exit()

# Remover colunas 'Unnamed: 0' e 'id_cliente' conforme instrução anterior ou inferência de irrelevância para o modelo
# 'Unnamed: 0' é geralmente um índice lido indevidamente.
# 'id_cliente' e 'data_ref' não são variáveis explicativas diretas para o modelo de renda em si,
# mas 'data_ref' pode ser útil para outras análises temporais no projeto final.
# Para este exercício, vamos focar nas variáveis descritas na tabela.
df = df.drop(columns=['Unnamed: 0', 'id_cliente'], errors='ignore')

# 1. Preparar a base de dados
# Remover linhas duplicadas
initial_rows = len(df)
df.drop_duplicates(inplace=True)
print(f"Número de linhas após remover duplicatas: {len(df)} (Original: {initial_rows})")

# Tratar valores nulos (NaN)
print("\nValores Nulos por Coluna antes do tratamento:")
print(df.isnull().sum())

# Remover NaNs da variável 'tempo_emprego' e quaisquer outras variáveis que se tornarem cruciais para o modelo
# 'renda' já não tem NaNs na info() fornecida, mas é bom garantir que seja > 0 para o log.
df_clean = df.dropna(subset=['tempo_emprego']).copy()
df_clean = df_clean[df_clean['tempo_emprego'] > 0].copy()
df_clean = df_clean[df_clean['renda'] > 0].copy() # Garantir renda > 0 para log

print(f"\nNúmero de linhas após remover NaNs e valores <= 0 em 'tempo_emprego'/'renda': {len(df_clean)}")

# Criar a variável log_renda
df_clean['log_renda'] = np.log(df_clean['renda'])
print("\nVariável 'log_renda' criada.")

# Verificar se há NaNs restantes em outras colunas categóricas que Patsy possa não lidar bem
# Embora Patsy lide com NaNs, para modelos de regressão é melhor ter dados limpos.
# Para este exercício, assumimos que as colunas categóricas e numéricas que não 'tempo_emprego'
# e 'renda' estão completas ou que seus NaNs não afetam significativamente o ajuste após as remoções anteriores.
print("\nValores Nulos por Coluna após limpeza primária:")
print(df_clean.isnull().sum())
# Se houver NaNs em 'qt_pessoas_residencia', 'idade', etc., considere df_clean.dropna().


# 2. Ajustar um modelo para prever log(renda) considerando todas as covariáveis disponíveis.
# 3. Utilizando os recursos do Patsy, coloque as variáveis qualitativas como dummies.
#    Mantenha sempre a categoria mais frequente como casela de referência.
#    Patsy faz isso automaticamente quando você usa smf.ols e fornece a fórmula.
#    A categoria mais frequente é a padrão para referência.

print("\n--- Ajustando o Modelo de Regressão Múltipla Completo ---")

# Listar todas as variáveis explicativas (exceto as que já foram tratadas ou são a variável resposta/ID/Data)
# 'data_ref' e 'renda' são a resposta ou meta.
# 'Unnamed: 0' e 'id_cliente' foram removidos.
explanatory_vars = [
    'sexo', 'posse_de_veiculo', 'posse_de_imovel', 'qtd_filhos',
    'tipo_renda', 'educacao', 'estado_civil', 'tipo_residencia',
    'idade', 'tempo_emprego', 'qt_pessoas_residencia'
]

# Construir a fórmula do Patsy
# C() para garantir que Patsy trate como categórica, se houver dúvida.
# Embora para 'object' e 'bool' ele geralmente já faça isso.
# Para 'tempo_emprego' sem log, pois não foi pedido para transformar nesta fase.
# I() para variáveis booleanas como posse_de_veiculo e posse_de_imovel, se Patsy interpretar como numéricas.
# No entanto, Patsy é inteligente o suficiente para `bool` e `object`.
formula_completa = 'log_renda ~ ' + ' + '.join(explanatory_vars)

print(f"\nFórmula do modelo:\n{formula_completa}")

# Ajustar o modelo
model_completo = smf.ols(formula_completa, data=df_clean).fit()

print("\n--- Sumário do Modelo de Regressão Múltipla Completo ---")
print(model_completo.summary())


# 4. Avalie os parâmetros e veja se parecem fazer sentido prático.
print("\n--- Avaliação e Interpretação dos Parâmetros ---")
print("Os coeficientes representam o impacto na **log(renda)**. Para entender o efeito na **renda** original, podemos usar $e^{\beta}-1$ para um efeito percentual.")

for param, coef in model_completo.params.drop('Intercept', errors='ignore').items():
    p_value = model_completo.pvalues.get(param, np.nan) # Pegar p-valor, se existir
    is_significant = " (Significante)" if p_value < 0.05 else " (NÃO Significante)"

    # Tratamento especial para variáveis contínuas (idade, tempo_emprego, qtd_filhos, qt_pessoas_residencia)
    if param in ['idade', 'qtd_filhos', 'qt_pessoas_residencia']:
        percent_effect = (np.exp(coef) - 1) * 100
        print(f"- **{param}**: Um aumento de uma unidade em {param} está associado a uma mudança de aproximadamente **{percent_effect:.2f}% na renda**, mantendo outras variáveis constantes.{is_significant}")
        if param == 'idade':
            print(f"  *Sentido prático:* Geralmente, a renda aumenta com a idade até um certo ponto. Um coeficiente positivo faz sentido.")
        elif param == 'qtd_filhos':
            print(f"  *Sentido prático:* O número de filhos pode ter efeitos variados na renda. Um coeficiente negativo pode indicar custos ou tempo dedicados à família que afetam a renda, enquanto um positivo pode ser associado a estabilidade familiar. Observe o sinal.")
        elif param == 'qt_pessoas_residencia':
            print(f"  *Sentido prático:* Mais pessoas na residência podem indicar mais contribuintes para a renda (se forem adultos) ou mais dependentes. Observe o sinal.")
    elif param == 'tempo_emprego': # Sem transformação log
        percent_effect = (np.exp(coef) - 1) * 100
        print(f"- **{param}**: Um aumento de uma unidade (um ano) no tempo de emprego está associado a uma mudança de aproximadamente **{percent_effect:.2f}% na renda**, mantendo outras variáveis constantes.{is_significant}")
        print(f"  *Sentido prático:* Geralmente, a renda tende a aumentar com mais tempo de emprego devido a experiência e progressão na carreira. Um coeficiente positivo é esperado.")
    elif '[T.' in param: # Variáveis categóricas (dummies)
        var_name = param.split('[T.')[0]
        category = param.split('[T.')[1].replace(']', '')
        percent_effect = (np.exp(coef) - 1) * 100
        print(f"- **{var_name} (Categoria: {category})**: Estar nesta categoria, em comparação com a categoria de referência (mais frequente), está associado a uma mudança de aproximadamente **{percent_effect:.2f}% na renda**, mantendo outras variáveis constantes.{is_significant}")
        if var_name == 'sexo':
            print(f"  *Sentido prático:* Diferenças de renda por sexo são comuns e podem indicar disparidades salariais. Observe se a categoria de referência é 'M' ou 'F'.")
        elif var_name == 'posse_de_veiculo':
            print(f"  *Sentido prático:* Possuir um veículo pode estar associado a uma maior capacidade financeira ou necessidade para o trabalho, o que pode influenciar a renda.")
        elif var_name == 'posse_de_imovel':
            print(f"  *Sentido prático:* Semelhante ao veículo, possuir imóvel pode ser um indicador de maior estabilidade financeira e, consequentemente, renda mais alta.")
        elif var_name == 'tipo_renda':
            print(f"  *Sentido prático:* Diferentes tipos de renda (ex: 'Assalariado', 'Empresário') podem ter níveis de renda associados distintos.")
        elif var_name == 'educacao':
            print(f"  *Sentido prático:* O nível de educação é um forte preditor de renda na maioria dos contextos. Espera-se que níveis de educação mais altos estejam associados a rendas maiores.")
        elif var_name == 'estado_civil':
            print(f"  *Sentido prático:* O estado civil pode estar relacionado a dinâmicas de renda familiar ou estabilidade. Pode ter efeitos variados.")
        elif var_name == 'tipo_residencia':
            print(f"  *Sentido prático:* O tipo de residência pode refletir o padrão de vida e, indiretamente, a renda.")
    else: # Outras variáveis que não se encaixam nos padrões acima
        print(f"- **{param}**: Coeficiente: {coef:.4f}. Este é um coeficiente linear direto na log_renda. O efeito percentual na renda é {(np.exp(coef) - 1) * 100:.2f}%.{is_significant}")

print("\n**Observações sobre os p-valores e significância:**")
print("- Os p-valores no sumário (`P>|t|`) indicam a significância estatística de cada covariável. Um p-valor abaixo de um limite (comumente 0.05) sugere que a variável é estatisticamente significante para explicar a `log_renda`.")
print("- Variáveis não significantes podem ser removidas para simplificar o modelo, como fizemos no exercício anterior.")
print("- O **Intercepto** representa o logaritmo da renda esperada quando todas as variáveis explicativas contínuas são zero e todas as variáveis categóricas estão na sua categoria de referência.")
print("\n**Em resumo, a maioria dos parâmetros deve fazer sentido prático com base no conhecimento de fatores que afetam a renda.**")
print("Por exemplo, 'tempo_emprego' e 'idade' tendem a ter coeficientes positivos, e níveis de 'educacao' mais altos tendem a ter coeficientes positivos em relação à categoria de referência.")

--- Exercício: Previsão de Renda - Regressão Múltipla ---
Base de dados 'previsao_de_renda.csv' carregada com sucesso!
Número de linhas após remover duplicatas: 14593 (Original: 15000)

Valores Nulos por Coluna antes do tratamento:
data_ref                    0
sexo                        0
posse_de_veiculo            0
posse_de_imovel             0
qtd_filhos                  0
tipo_renda                  0
educacao                    0
estado_civil                0
tipo_residencia             0
idade                       0
tempo_emprego            2503
qt_pessoas_residencia       0
renda                       0
dtype: int64

Número de linhas após remover NaNs e valores <= 0 em 'tempo_emprego'/'renda': 12090

Variável 'log_renda' criada.

Valores Nulos por Coluna após limpeza primária:
data_ref                 0
sexo                     0
posse_de_veiculo         0
posse_de_imovel          0
qtd_filhos               0
tipo_renda               0
educacao                 0
estado_civil

In [11]:
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrices
import matplotlib.pyplot as plt
import warnings

# Ignorar warnings para melhor visualização (cuidado ao usar em produção)
warnings.filterwarnings('ignore')

print("--- Refinamento do Modelo de Previsão de Renda: Remoção da Variável Menos Significante ---")

# Re-carregar e preparar a base de dados (garantindo que estamos no mesmo ponto de antes)
try:
    df = pd.read_csv('previsao_de_renda.csv')
    df = df.drop(columns=['Unnamed: 0', 'id_cliente'], errors='ignore')
    df.drop_duplicates(inplace=True)
    df_clean = df.dropna(subset=['tempo_emprego']).copy()
    df_clean = df_clean[df_clean['tempo_emprego'] > 0].copy()
    df_clean = df_clean[df_clean['renda'] > 0].copy()
    df_clean['log_renda'] = np.log(df_clean['renda'])

    df_multi_model = df_clean.dropna(subset=[
        'sexo', 'posse_de_veiculo', 'posse_de_imovel', 'qtd_filhos',
        'tipo_renda', 'educacao', 'estado_civil', 'tipo_residencia', 'idade',
        'qt_pessoas_residencia'
    ]).copy()
    print(f"Base de dados preparada. {len(df_multi_model)} linhas para modelagem múltipla.")

except Exception as e:
    print(f"Erro na preparação da base de dados: {e}")
    exit()

# Fórmula do modelo completo (como a última vez)
initial_formula_multi = (
    'log_renda ~ sexo + posse_de_veiculo + posse_de_imovel + qtd_filhos + '
    'tipo_renda + educacao + estado_civil + tipo_residencia + idade + '
    'tempo_emprego + qt_pessoas_residencia'
)

# Ajustar o modelo completo para ter os p-valores originais para comparação
model_completo = smf.ols(initial_formula_multi, data=df_multi_model).fit()
print("\n--- Sumário do Modelo Completo Original ---")
print(model_completo.summary())

# 1. Identificar a variável menos significante
p_values = model_completo.pvalues.drop('Intercept', errors='ignore')
# Garantir que estamos olhando apenas para variáveis que não sejam NaN ou Inf
p_values = p_values.replace([np.inf, -np.inf], np.nan).dropna()

if p_values.empty:
    print("\nNão há variáveis com p-valores válidos para remover. O modelo já pode ser o mais simples ou ter problemas.")
    least_significant_var = None
    max_p_value = None
else:
    least_significant_var = p_values.idxmax()
    max_p_value = p_values.max()
    print(f"\nVariável menos significante (maior p-valor): **{least_significant_var}** (P-valor: {max_p_value:.4f})")

# 2. Remover a variável da fórmula (se houver uma para remover)
if least_significant_var:
    explanatory_vars_list = [term.strip() for term in initial_formula_multi.split('~')[1].split('+')]

    # Lógica para remover a variável base de uma dummy, se for o caso
    var_to_remove_from_formula = None
    for original_var in ['sexo', 'posse_de_veiculo', 'posse_de_imovel', 'tipo_renda', 'educacao', 'estado_civil', 'tipo_residencia']:
        if least_significant_var.startswith(original_var):
            var_to_remove_from_formula = original_var
            break

    if var_to_remove_from_formula is None: # Não é uma dummy, ou o nome base já é o p-valor mais alto
        var_to_remove_from_formula = least_significant_var

    new_explanatory_vars = [v for v in explanatory_vars_list if v != var_to_remove_from_formula]

    if not new_explanatory_vars:
        new_formula = "log_renda ~ 1" # Apenas intercepto
    else:
        new_formula = 'log_renda ~ ' + ' + '.join(new_explanatory_vars)

    print(f"Nova fórmula após remover '{var_to_remove_from_formula}':\n{new_formula}")

    # 3. Reajustar o modelo com a nova fórmula
    model_reduzido = smf.ols(new_formula, data=df_multi_model).fit()

    print("\n--- Sumário do Modelo Reduzido (Após remover a variável menos significante) ---")
    print(model_reduzido.summary())

    # ---
    ### Avaliação da Melhoria do Modelo (Gerado Programaticamente)
    print("\n--- Avaliação da Melhoria do Modelo ---")

    # Coletar os indicadores
    r2_adj_completo = model_completo.rsquared_adj
    aic_completo = model_completo.aic
    bic_completo = model_completo.bic

    r2_adj_reduzido = model_reduzido.rsquared_adj
    aic_reduzido = model_reduzido.aic
    bic_reduzido = model_reduzido.bic

    # Determinar a avaliação
    eval_r2_adj = ''
    if r2_adj_reduzido > r2_adj_completo:
        eval_r2_adj = 'Melhorou (aumentou)'
    elif r2_adj_reduzido < r2_adj_completo:
        eval_r2_adj = 'Piorou (diminuiu)'
    else:
        eval_r2_adj = 'Permaneceu o mesmo'

    eval_aic = ''
    if aic_reduzido < aic_completo:
        eval_aic = 'Melhorou (menor)'
    elif aic_reduzido > aic_completo:
        eval_aic = 'Piorou (maior)'
    else:
        eval_aic = 'Permaneceu o mesmo'

    eval_bic = ''
    if bic_reduzido < bic_completo:
        eval_bic = 'Melhorou (menor)'
    elif bic_reduzido > bic_completo:
        eval_bic = 'Piorou (maior)'
    else:
        eval_bic = 'Permaneceu o mesmo'

    # Imprimir a tabela de comparação
    print(f"| Indicador           | Modelo Completo (Original) | Modelo Reduzido (Variável removida: {var_to_remove_from_formula}) | Avaliação            |")
    print("| :------------------ | :------------------------- | :---------------------------------------------------------------------- | :------------------- |")
    print(f"| **R-quadrado Ajustado** | {r2_adj_completo:.4f}             | {r2_adj_reduzido:.4f}                                                 | {eval_r2_adj:<20} |")
    print(f"| **AIC** | {aic_completo:.2f}                 | {aic_reduzido:.2f}                                                    | {eval_aic:<20}     |")
    print(f"| **BIC** | {bic_completo:.2f}                 | {bic_reduzido:.2f}                                                    | {eval_bic:<20}     |")


    print("\n**Interpretação da Melhoria:**")
    print("- **R-quadrado Ajustado**: Um aumento (ou queda muito pequena) indica que a variável removida não contribuía significativamente para explicar a variação da renda e que o modelo está mais conciso.")
    print("- **AIC e BIC**: Valores **menores** para AIC e BIC são desejáveis. Se esses valores diminuíram, o modelo ficou mais 'parcimonioso', ou seja, ele explica os dados tão bem (ou quase) com menos complexidade. Isso é geralmente um sinal de melhoria, especialmente para modelos que se buscam mais simples e generalizáveis.")

    print("\n--- Análise de Alteração dos Parâmetros ---")

    # Criação de DataFrames para facilitar a comparação
    params_completo = model_completo.params.to_frame(name='Coef. Modelo Completo')
    pvalues_completo = model_completo.pvalues.to_frame(name='P-valor Modelo Completo')

    params_reduzido = model_reduzido.params.to_frame(name='Coef. Modelo Reduzido')
    pvalues_reduzido = model_reduzido.pvalues.to_frame(name='P-valor Modelo Reduzido')

    # Combinar os DataFrames para uma visualização lado a lado
    # Usar .align para garantir que as linhas se alinhem, preenchendo com NaN onde não houver correspondência
    comparison_df = pd.concat([params_completo, pvalues_completo, params_reduzido, pvalues_reduzido], axis=1, join='outer')

    print(comparison_df.round(4).fillna('N/A').to_string()) # Usar to_string() para exibir todas as linhas/colunas, preencher NaNs

    print("\n**Análise:**")
    print(f"- A linha para **'{var_to_remove_from_formula}'** no 'Modelo Completo' agora tem 'N/A' nos campos do 'Modelo Reduzido', confirmando sua remoção.")
    print("- Para as variáveis que permaneceram, observe se houve **mudanças significativas nos coeficientes (magnitudes)**. Pequenas variações são esperadas, mas grandes alterações podem sugerir que a variável removida estava 'confundindo' os efeitos das outras variáveis, possivelmente devido à **multicolinearidade**.")
    print("- Verifique também se o **sinal dos coeficientes** das variáveis remanescentes mudou. Uma mudança de sinal é um alerta e pode indicar um problema sério no modelo ou na relação entre as variáveis.")
    print("- Observe os **p-valores** das variáveis restantes no 'Modelo Reduzido'. Idealmente, elas devem permanecer significantes (p < 0.05).")

else:
    print("\nNenhuma variável para remover ou problemas com p-valores. Não foi possível ajustar o modelo reduzido para comparação.")

--- Refinamento do Modelo de Previsão de Renda: Remoção da Variável Menos Significante ---
Base de dados preparada. 12090 linhas para modelagem múltipla.

--- Sumário do Modelo Completo Original ---
                            OLS Regression Results                            
Dep. Variable:              log_renda   R-squared:                       0.357
Model:                            OLS   Adj. R-squared:                  0.356
Method:                 Least Squares   F-statistic:                     279.6
Date:                Sat, 12 Jul 2025   Prob (F-statistic):               0.00
Time:                        14:44:21   Log-Likelihood:                -13214.
No. Observations:               12090   AIC:                         2.648e+04
Df Residuals:                   12065   BIC:                         2.666e+04
Df Model:                          24                                         
Covariance Type:            nonrobust                                         
           

In [13]:
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrices
import matplotlib.pyplot as plt
import warnings

# Ignorar warnings para melhor visualização
warnings.filterwarnings('ignore')

print("--- Otimização do Modelo de Previsão de Renda: Remoção Iterativa de Variáveis ---")

# Re-carregar e preparar a base de dados (garantindo consistência)
try:
    df = pd.read_csv('previsao_de_renda.csv')
    df = df.drop(columns=['Unnamed: 0', 'id_cliente'], errors='ignore')
    df.drop_duplicates(inplace=True)
    df_clean = df.dropna(subset=['tempo_emprego']).copy()
    df_clean = df_clean[df_clean['tempo_emprego'] > 0].copy()
    df_clean = df_clean[df_clean['renda'] > 0].copy()
    df_clean['log_renda'] = np.log(df_clean['renda'])

    df_multi_model = df_clean.dropna(subset=[
        'sexo', 'posse_de_veiculo', 'posse_de_imovel', 'qtd_filhos',
        'tipo_renda', 'educacao', 'estado_civil', 'tipo_residencia', 'idade',
        'qt_pessoas_residencia'
    ]).copy()
    print(f"Base de dados preparada. {len(df_multi_model)} linhas para modelagem múltipla.")

except Exception as e:
    print(f"Erro na preparação da base de dados: {e}")
    exit()

# Definir a fórmula inicial do modelo completo
initial_formula = (
    'log_renda ~ sexo + posse_de_veiculo + posse_de_imovel + qtd_filhos + '
    'tipo_renda + educacao + estado_civil + tipo_residencia + idade + '
    'tempo_emprego + qt_pessoas_residencia'
)

# Ajustar e guardar o modelo inicial para comparação
print("\n--- Sumário do Modelo Inicial (Completo) ---")
model_initial = smf.ols(initial_formula, data=df_multi_model).fit()
print(model_initial.summary())

# ---
# Processo de Remoção Iterativa de Variáveis
# ---
print("\n--- Início do Processo de Eliminação Backward (Remoção Iterativa) ---")

current_formula = initial_formula
iterations = 0

while True:
    iterations += 1
    model_current = smf.ols(current_formula, data=df_multi_model).fit()
    p_values = model_current.pvalues.drop('Intercept', errors='ignore') # Ignorar o intercepto

    # Filtrar p-valores válidos e não significantes
    p_values = p_values.replace([np.inf, -np.inf], np.nan).dropna()
    non_significant_vars = p_values[p_values >= 0.05] # >= 0.05 para incluir 0.05 como não significante

    if non_significant_vars.empty:
        print(f"\nIteração {iterations}: Todas as variáveis restantes são significantes (p < 0.05). Processo encerrado.")
        break # Sai do loop se todas as variáveis forem significantes
    else:
        # Encontrar a variável com o maior p-valor não significante
        var_to_remove = non_significant_vars.idxmax()
        max_p_value = non_significant_vars.max()

        print(f"\nIteração {iterations}: Removendo **{var_to_remove}** (P-valor: {max_p_value:.4f})")

        # Lógica para remover a variável base da fórmula
        # Primeiro, obter a lista de variáveis explicativas da fórmula atual
        explanatory_vars_in_formula = [term.strip() for term in current_formula.split('~')[1].split('+')]

        # Descobrir o nome da variável 'base' se for uma dummy (ex: 'sexo[T.M]' -> 'sexo')
        base_var_name_to_remove = None
        for original_var in ['sexo', 'posse_de_veiculo', 'posse_de_imovel', 'tipo_renda', 'educacao', 'estado_civil', 'tipo_residencia']:
            if var_to_remove.startswith(original_var):
                base_var_name_to_remove = original_var
                break
        
        if base_var_name_to_remove is None: # Se não for uma variável categórica com dummy, usa o nome exato
            base_var_name_to_remove = var_to_remove

        # Criar a nova lista de variáveis explicativas, excluindo a variável a ser removida
        new_explanatory_vars_list = [v for v in explanatory_vars_in_formula if v != base_var_name_to_remove]

        if not new_explanatory_vars_list: # Se a lista ficar vazia, significa que sobrou apenas o intercepto
            current_formula = "log_renda ~ 1"
            print(f"Nova fórmula: {current_formula} (apenas intercepto)")
            break # Nenhuma variável explicativa restante, parar
        else:
            current_formula = 'log_renda ~ ' + ' + '.join(new_explanatory_vars_list)
            print(f"Nova fórmula: {current_formula}")

# O modelo `model_current` no final do loop será o nosso modelo final.
final_model = model_current
print("\n--- Sumário do Modelo Final (Todas as Variáveis Significantess) ---")
print(final_model.summary())

# ---
# Comparação Final: Modelo Inicial vs. Modelo Final
# ---
print("\n--- Comparação Final: Modelo Inicial vs. Modelo Final ---")

# Coletar indicadores
r2_adj_initial = model_initial.rsquared_adj
aic_initial = model_initial.aic
bic_initial = model_initial.bic

r2_adj_final = final_model.rsquared_adj
aic_final = final_model.aic
bic_final = final_model.bic

# Determinar a avaliação
eval_r2_adj = ''
if r2_adj_final > r2_adj_initial:
    eval_r2_adj = 'Melhorou (aumentou)'
elif r2_adj_final < r2_adj_initial:
    eval_r2_adj = 'Piorou (diminuiu)'
else:
    eval_r2_adj = 'Permaneceu o mesmo'

eval_aic = ''
if aic_final < aic_initial:
    eval_aic = 'Melhorou (menor)'
elif aic_final > aic_initial:
    eval_aic = 'Piorou (maior)'
else:
    eval_aic = 'Permaneceu o mesmo'

eval_bic = ''
if bic_final < bic_initial:
    eval_bic = 'Melhorou (menor)'
elif bic_final > bic_initial:
    eval_bic = 'Piorou (maior)'
else:
    eval_bic = 'Permaneceu o mesmo'

# Imprimir a tabela de comparação
print(f"| Indicador           | Modelo Inicial (Completo) | Modelo Final (Otimizado) | Avaliação              |")
print("| :------------------ | :------------------------ | :----------------------- | :--------------------- |")
print(f"| **R-quadrado Ajustado** | {r2_adj_initial:.4f}              | {r2_adj_final:.4f}               | {eval_r2_adj:<22} |")
print(f"| **AIC** | {aic_initial:.2f}                 | {aic_final:.2f}                  | {eval_aic:<22}     |")
print(f"| **BIC** | {bic_initial:.2f}                 | {bic_final:.2f}                  | {eval_bic:<22}     |")

print("\n--- Conclusão sobre o Modelo Final ---")
print("O objetivo da remoção iterativa é encontrar um modelo mais **parcimonioso** (mais simples, com menos variáveis) que ainda assim mantenha um bom poder explicativo ou até o melhore.")
print(f"\nCom base nos indicadores:")
if r2_adj_final >= r2_adj_initial and (aic_final <= aic_initial or bic_final <= bic_initial):
    print("- O **R-quadrado Ajustado** do modelo final {eval_r2_adj.lower()} ou {r2_adj_final:.4f}, indicando que ele explica a variância da `log_renda` tão bem ou melhor, mesmo com menos variáveis.")
    print("- Os valores de **AIC e BIC** do modelo final {eval_aic.lower()} e {eval_bic.lower()} respectivamente. **Valores menores** para AIC e BIC são preferíveis, pois indicam um modelo que balanceia bem o ajuste e a complexidade.")
    print("\n**Conclusão**: O modelo final **parece ser melhor** que o modelo inicial. Ele é mais simples (tem menos variáveis), o que facilita a interpretação e reduz o risco de overfitting (ajuste excessivo aos dados de treinamento). A remoção de variáveis não significantes eliminou 'ruído' e melhorou a eficiência do modelo sem comprometer (e talvez até melhorando) sua capacidade de generalização.")
else:
    print("- O **R-quadrado Ajustado** do modelo final {eval_r2_adj.lower()} para {r2_adj_final:.4f}.")
    print("- Os valores de **AIC e BIC** do modelo final {eval_aic.lower()} e {eval_bic.lower()} respectivamente.")
    print("\n**Conclusão**: A avaliação do modelo final em relação ao inicial é mista. Embora a remoção de variáveis tenha simplificado o modelo, os indicadores podem não ter melhorado universalmente. Nesses casos, a escolha do 'melhor' modelo pode depender do equilíbrio desejado entre simplicidade e poder preditivo.")

--- Otimização do Modelo de Previsão de Renda: Remoção Iterativa de Variáveis ---
Base de dados preparada. 12090 linhas para modelagem múltipla.

--- Sumário do Modelo Inicial (Completo) ---
                            OLS Regression Results                            
Dep. Variable:              log_renda   R-squared:                       0.357
Model:                            OLS   Adj. R-squared:                  0.356
Method:                 Least Squares   F-statistic:                     279.6
Date:                Sat, 12 Jul 2025   Prob (F-statistic):               0.00
Time:                        14:45:47   Log-Likelihood:                -13214.
No. Observations:               12090   AIC:                         2.648e+04
Df Residuals:                   12065   BIC:                         2.666e+04
Df Model:                          24                                         
Covariance Type:            nonrobust                                         
                   